In [1]:
import numpy as np
import torch

SEED = 1234
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

# 0. Choosing "teacher" model

In [2]:
checkpoint = "bert-base-uncased"
task_name  = "mrpc"

# 1. Loading our mrpc part of the GLUE dataset

In [3]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

raw_datasets = load_dataset("glue", task_name)
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

In [5]:
raw_datasets['train']

Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 3668
})

In [6]:
from datasets import DatasetDict

In [7]:
raw_datasets = DatasetDict({
    "train": raw_datasets['train'],
    "validation": raw_datasets['validation'],
    "test": raw_datasets['test']
})

In [8]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

In [9]:
raw_train_dataset = raw_datasets["train"]
raw_train_dataset[0]

{'sentence1': 'Amrozi accused his brother , whom he called " the witness " , of deliberately distorting his evidence .',
 'sentence2': 'Referring to him as only " the witness " , Amrozi accused his brother of deliberately distorting his evidence .',
 'label': 1,
 'idx': 0}

In [10]:
raw_train_dataset[5]['sentence1'], raw_train_dataset[5]['sentence2']

('Revenue in the first quarter of the year dropped 15 percent from the same period a year earlier .',
 "With the scandal hanging over Stewart 's company , revenue the first quarter of the year dropped 15 percent from the same period a year earlier .")

In [11]:
raw_train_dataset[5]['label']

1

In [12]:
raw_train_dataset[5]['idx']

5

In [13]:
raw_train_dataset.features

{'sentence1': Value('string'),
 'sentence2': Value('string'),
 'label': ClassLabel(names=['not_equivalent', 'equivalent']),
 'idx': Value('int32')}

# 2. Preprocess

In [14]:
def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)

tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1725
    })
})

# 3. Preparing for Training

In [15]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [16]:
tokenized_datasets = tokenized_datasets.remove_columns(["sentence1", "sentence2", "idx"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")
tokenized_datasets["train"].column_names

['labels', 'input_ids', 'token_type_ids', 'attention_mask']

In [17]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(
    tokenized_datasets["train"], shuffle=True, batch_size=32, collate_fn=data_collator
)

val_dataloader = DataLoader(
    tokenized_datasets["validation"], batch_size=32, collate_fn=data_collator
)

eval_dataloader = DataLoader(
    tokenized_datasets["test"], batch_size=32, collate_fn=data_collator
)

In [18]:
for batch in train_dataloader:
    break
{k: v.shape for k, v in batch.items()}

{'labels': torch.Size([32]),
 'input_ids': torch.Size([32, 100]),
 'token_type_ids': torch.Size([32, 100]),
 'attention_mask': torch.Size([32, 100])}

# 4. Load Model

In [19]:
import sys
sys.path.insert(0, '../')

In [20]:
from Bert_model.modeling_bert import BertForSequenceClassification

In [21]:
# id2label, label2id dicts for the outputs for the model
labels = tokenized_datasets["train"].features["labels"].names
num_labels = len(labels)
label2id, id2label = dict(), dict()
for i, label in enumerate(labels):
    label2id[label] = str(i)
    id2label[str(i)] = label

In [22]:
model = BertForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    output_hidden_states=True,
    output_attentions=True
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [23]:
model.set_use_module_grafting(False)
model.set_use_scc_status(False)

In [24]:
outputs = model(**batch)
print(outputs.loss, outputs.logits.shape)

tensor(1.2996, grad_fn=<NllLossBackward0>) torch.Size([32, 2])


In [25]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)

device

device(type='cuda')

### Load Trained Weights

In [28]:
load_path = '../glue_fine_tune/weights/'
best_weight = torch.load(load_path + f'bert-{task_name}.pt')
model.load_state_dict(best_weight['model_state_dict'])

<All keys matched successfully>

In [29]:
from train_eval_func import eval_loop

In [30]:
eval_loop(model, val_dataloader, task_name, device)[0]

{'accuracy': 0.3161764705882353, 'f1': 0.0}

# 5. Merge Most Similar Layers

In [31]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sn
from tqdm import tqdm

In [32]:
import os
from merge_helper import merge_bert_layers, LayerMergeTracker
from train_eval_func import train, get_primary_metric, set_lr_scheduler
from torch.optim import AdamW

In [33]:
init_metric = eval_loop(model, val_dataloader, task_name, device)[0]
init_metric

{'accuracy': 0.3161764705882353, 'f1': 0.0}

In [34]:
tracker = LayerMergeTracker(num_layers=12)

print("Initial state:")
print(f"  {tracker.get_mapping()}")
print(f"  Total layers: {len(tracker)}")
print()

Initial state:
  [[0], [1], [2], [3], [4], [5], [6], [7], [8], [9], [10], [11]]
  Total layers: 12



In [35]:
from CKA import CKAEvaluator

In [36]:
cka_eval = CKAEvaluator(device)

In [36]:
path = './'

In [ ]:
orig_n_layer = len(model.bert.encoder.layer)

target_metric = get_primary_metric(task_name)
print(f"Target Metric for {task_name}: {target_metric}")

threshold = init_metric[target_metric] * 0.01
print(f"Original {target_metric} score: {init_metric[target_metric]}, Threshold: {threshold}")
print("")

for merge_itr in range(0, 6):
    print("-------> Merge Iteration:", merge_itr + 1)

    # Layer Similarity Estimation
    cls_reps_similarity = cka_eval.pairwise(model, train_dataloader, device, only_cls_token=True, max_iter=float("Inf"))
    stats = cka_eval.similarity_stats(cls_reps_similarity)
    print("///// Similarity Stats /////")
    print("Average Similarity:", stats['average'])
    print("Highest Similarity:", stats['max'])
    print("Lowest Similarity:", stats['min'])

    # Merge Most Adjacent Similar Layer Pair
    merge_layer = stats['argmax']
    merged = merge_bert_layers(model, merge_layer, merge_layer + 1)
    model.bert.encoder.layer[merge_layer] = merged
    del model.bert.encoder.layer[merge_layer + 1]

    print("///// Merge Layer /////")
    print(f"Merged Layers: {merge_layer}, {merge_layer + 1}")
    
    tracker.merge(merge_layer)
    print("///// Layer Composition /////")
    print(f"  After:  {tracker.get_mapping()}")
    print(f"  Total layers: {len(tracker)}")

    # Evaluate Impacted Performance
    eval_metric = eval_loop(model, val_dataloader, task_name, device)[0]
    print("///// Impacted Performance /////")
    for metric in eval_metric.keys():
        print(f"{metric}: {eval_metric[metric]}", end=" ")
    print("")

    # Recovery Training
    print("///// Recovery Training /////")
    diff = init_metric[target_metric] - eval_metric[target_metric]
    if diff > threshold: 
        # Setup for recovery training
        recovery_epochs = 3
        recovery_lr = 1e-5

        # Create optimizer
        optimizer = AdamW(model.parameters(), lr=recovery_lr)
        
        # Create scheduler
        num_training_steps = recovery_epochs * len(train_dataloader)
        lr_scheduler = set_lr_scheduler(
            optimizer=optimizer,
            num_training_steps=num_training_steps
        )
        
        print(f"Training: {recovery_epochs} epochs, lr={recovery_lr}, steps={num_training_steps}")

        # Train with scheduler
        train_stats = train(
            model, 
            train_dataloader, 
            val_dataloader, 
            task_name, 
            device,
            num_epochs=recovery_epochs,
            lr=recovery_lr,
            patience=2,
            use_early_stopping=True,
            optimizer=optimizer,
            lr_scheduler=lr_scheduler,
            display_epoch_iter=True
        )
        
        n_layer = orig_n_layer - merge_itr - 1
        if (n_layer == 6) or (n_layer == 7) or (n_layer == 8):
            
            saveObject = {
                'model_state_dict': model.state_dict(),
                'layer_track': tracker.get_mapping(),
                'train_stats': train_stats
            }
            torch.save(saveObject, path + f'layer-{n_layer}-{task_name}.pt')   
    else:
        print("No need for recovery training")
    
    print("")

Target Metric for mrpc: f1
Original f1 score: 0.9048414023372288, Threshold: 0.009048414023372288

-------> Merge Iteration: 1


CKA Evaluation: 100%|████████████████████████████████████████████████████████████████| 115/115 [00:44<00:00,  2.57it/s]


///// Similarity Stats /////
Average Similarity: 0.9062627553939819
Highest Similarity: 0.9965952038764954
Lowest Similarity: 0.7515758275985718
///// Merge Layer /////
Merged Layers: 9, 10
///// Layer Composition /////
  After:  [[0], [1], [2], [3], [4], [5], [6], [7], [8], [9, 10], [11]]
  Total layers: 11
///// Impacted Performance /////
accuracy: 0.8627450980392157 f1: 0.9063545150501672 
///// Recovery Training /////
No need for recovery training

-------> Merge Iteration: 2


CKA Evaluation: 100%|████████████████████████████████████████████████████████████████| 115/115 [00:49<00:00,  2.32it/s]


///// Similarity Stats /////
Average Similarity: 0.895910382270813
Highest Similarity: 0.9798085689544678
Lowest Similarity: 0.7529017329216003
///// Merge Layer /////
Merged Layers: 6, 7
///// Layer Composition /////
  After:  [[0], [1], [2], [3], [4], [5], [6, 7], [8], [9, 10], [11]]
  Total layers: 10
///// Impacted Performance /////
accuracy: 0.8186274509803921 f1: 0.8817891373801917 
///// Recovery Training /////
Training: 3 epochs, lr=1e-05, steps=345


...

Training:   0%|          | 0/345 [00:00<?, ?it/s]


<----------------- Epoch 1 ----------------->
Loss: 0.21, Training Metrics:
  accuracy: 0.9171
  f1: 0.9378
Validation Loss: 0.46, Validation Metrics:
  accuracy: 0.8505
  f1: 0.8957
Elapsed Time: 216.9229 sec

<----------------- Epoch 2 ----------------->
Loss: 0.14, Training Metrics:
  accuracy: 0.9512
  f1: 0.9636
Validation Loss: 0.48, Validation Metrics:
  accuracy: 0.8505
  f1: 0.8924
Elapsed Time: 215.9581 sec

<----------------- Epoch 3 ----------------->
Loss: 0.09, Training Metrics:
  accuracy: 0.9706
  f1: 0.9780
Validation Loss: 0.56, Validation Metrics:
  accuracy: 0.8431
  f1: 0.8893
Elapsed Time: 219.8354 sec

Total Training Time: 652.72 sec

-------> Merge Iteration: 6



CKA Evaluation: 100%|████████████████████████████████████████████████████████████████| 115/115 [01:02<00:00,  1.84it/s]


///// Similarity Stats /////
Average Similarity: 0.790017306804657
Highest Similarity: 0.8805225491523743
Lowest Similarity: 0.6635132431983948
///// Merge Layer /////
Merged Layers: 5, 6
///// Layer Composition /////
  After:  [[0], [1, 2], [3], [4], [5, 6, 7], [8, 9, 10, 11]]
  Total layers: 6
///// Impacted Performance /////
accuracy: 0.8284313725490197 f1: 0.8801369863013698 
///// Recovery Training /////
Training: 3 epochs, lr=1e-05, steps=345


Training:   0%|          | 0/345 [00:00<?, ?it/s]

In [35]:
def iterative_cka_merge_and_train(
    model,
    train_dataloader,
    val_dataloader,
    task_name,
    device,
    cka_evaluator,
    tracker,
    init_metric,
    num_merges=6,
    target_layers=[6, 7, 8],
    recovery_epochs=3,
    recovery_lr=1e-5,
    patience=2,
    save_dir='./weights/',
    keep_temp_checkpoints=False
):
    """
    Iteratively merge BERT layers using CKA similarity and recover performance.
    
    Args:
        model: BERT model to merge
        train_dataloader: Training dataloader
        val_dataloader: Validation dataloader
        task_name: GLUE task name (e.g., "mrpc", "qnli", "sst2")
        device: torch device
        cka_evaluator: CKAEvaluator instance
        tracker: LayerMergeTracker instance
        init_metric: Initial metrics dict before any merging
        num_merges: Number of merge iterations (default: 6)
        target_layers: Layer counts to save checkpoints for (default: [6, 7, 8])
        recovery_epochs: Epochs for recovery training (default: 3)
        recovery_lr: Learning rate for recovery (default: 1e-5)
        patience: Early stopping patience (default: 2)
        save_dir: Directory to save final models (default: './weights/')
        keep_temp_checkpoints: Keep temporary recovery checkpoints (default: False)
    
    Returns:
        Dictionary with merge history
    """
    # Setup
    orig_n_layer = len(model.bert.encoder.layer)
    target_metric = get_primary_metric(task_name)
    threshold = init_metric[target_metric] * 0.01
    
    # Create recovery checkpoint directory
    recovery_dir = os.path.join(save_dir, 'recovery')
    os.makedirs(recovery_dir, exist_ok=True)
    os.makedirs(save_dir, exist_ok=True)
    
    print(f"Target Metric for {task_name}: {target_metric}")
    print(f"Original {target_metric} score: {init_metric[target_metric]:.4f}")
    print(f"Recovery threshold: {threshold:.4f}")
    print("")
    
    # History tracking
    merge_history = {
        'iterations': [],
        'layer_compositions': [],
        'metrics_after_merge': [],
        'metrics_after_recovery': [],
        'training_stats': []
    }
    
    # Merge loop
    for merge_itr in range(num_merges):
        print("=" * 70)
        print(f"Merge Iteration: {merge_itr + 1}/{num_merges}")
        print("=" * 70)
        
        # Layer Similarity Estimation
        print("\n[1/5] Computing CKA Similarity...")
        cls_reps_similarity = cka_evaluator.pairwise(
            model, 
            train_dataloader, 
            device, 
            only_cls_token=True, 
            max_iter=float("Inf")
        )
        stats = cka_evaluator.similarity_stats(cls_reps_similarity)
        
        print("  Similarity Stats:")
        print(f"    Average: {stats['average']:.6f}")
        print(f"    Highest: {stats['max']:.6f}")
        print(f"    Lowest:  {stats['min']:.6f}")
        
        # Merge Most Adjacent Similar Layer Pair
        print("\n[2/5] Merging Layers...")
        merge_layer = stats['argmax']
        merged = merge_bert_layers(model, merge_layer, merge_layer + 1)
        model.bert.encoder.layer[merge_layer] = merged
        del model.bert.encoder.layer[merge_layer + 1]
        
        tracker.merge(merge_layer)
        print(f"  Merged Layers: {merge_layer} + {merge_layer + 1}")
        print(f"  Layer Composition: {tracker.get_mapping()}")
        print(f"  Total Layers: {len(tracker)}")
        
        # Evaluate Impacted Performance
        print("\n[3/5] Evaluating Post-Merge Performance...")
        eval_metric = eval_loop(model, val_dataloader, task_name, device)[0]
        print("  Metrics:")
        for metric_name, value in eval_metric.items():
            marker = "★" if metric_name == target_metric else " "
            print(f"    {marker} {metric_name}: {value:.4f}")
        
        merge_history['metrics_after_merge'].append(eval_metric)
        
        # Recovery Training
        print("\n[4/5] Recovery Training...")
        diff = init_metric[target_metric] - eval_metric[target_metric]
        print(f"  Performance drop: {diff:.4f} (threshold: {threshold:.4f})")
        
        if diff > threshold:
            print("  → Recovery training NEEDED")
            
            # Setup optimizer and scheduler
            optimizer = AdamW(model.parameters(), lr=recovery_lr)
            num_training_steps = recovery_epochs * len(train_dataloader)
            lr_scheduler = set_lr_scheduler(
                optimizer=optimizer,
                num_training_steps=num_training_steps
            )
            
            print(f"  → Training: {recovery_epochs} epochs, lr={recovery_lr}")
            
            # Temporary checkpoint path
            temp_save_path = os.path.join(recovery_dir, f'temp_iter{merge_itr}.pt')
            
            # Train with checkpoint saving
            train_stats = train(
                model,
                train_dataloader,
                val_dataloader,
                task_name,
                device,
                num_epochs=recovery_epochs,
                lr=recovery_lr,
                patience=patience,
                use_early_stopping=True,
                optimizer=optimizer,
                lr_scheduler=lr_scheduler,
                save_path=temp_save_path,
                display_epoch_iter=True
            )
            
            # CRITICAL: Reload best checkpoint
            print("\n  → Loading best checkpoint...")
            best_checkpoint = torch.load(temp_save_path)
            model.load_state_dict(best_checkpoint['model_state_dict'])
            
            print(f"  ✓ Loaded best checkpoint from epoch {best_checkpoint['epoch']}")
            print(f"    Best val loss: {best_checkpoint['val_loss']:.4f}")
            print(f"    Best val metrics: {best_checkpoint['val_metrics']}")
            
            merge_history['metrics_after_recovery'].append(best_checkpoint['val_metrics'])
            merge_history['training_stats'].append(train_stats)
            
            # Clean up temp checkpoint if not keeping
            if not keep_temp_checkpoints:
                os.remove(temp_save_path)
                print(f"  ✓ Cleaned up temporary checkpoint")
        else:
            print("  → Recovery training NOT needed")
            merge_history['metrics_after_recovery'].append(eval_metric)
            merge_history['training_stats'].append(None)
        
        # Save final model at target layer counts
        print("\n[5/5] Saving Model...")
        n_layer = orig_n_layer - merge_itr - 1
        if n_layer in target_layers:
            save_path = os.path.join(save_dir, f'layer-{n_layer}-{task_name}.pt')
            
            save_object = {
                'model_state_dict': model.state_dict(),
                'layer_track': tracker.get_mapping(),
                'train_stats': merge_history['training_stats'][-1],
                'metrics_after_merge': merge_history['metrics_after_merge'][-1],
                'metrics_after_recovery': merge_history['metrics_after_recovery'][-1],
                'merge_iteration': merge_itr + 1,
                'num_layers': n_layer
            }
            
            torch.save(save_object, save_path)
            print(f"  ✓ Saved {n_layer}-layer model to {save_path}")
        else:
            print(f"  • {n_layer} layers (not in target list, skipping save)")
        
        # Store iteration info
        merge_history['iterations'].append({
            'iteration': merge_itr + 1,
            'merged_layers': (merge_layer, merge_layer + 1),
            'num_layers_after': n_layer
        })
        merge_history['layer_compositions'].append(tracker.get_mapping())
        
        print("")
    
    # Final summary
    print("=" * 70)
    print("MERGE LOOP COMPLETED")
    print("=" * 70)
    print(f"Total merges: {num_merges}")
    print(f"Final layers: {len(tracker)}")
    print(f"Final composition: {tracker.get_mapping()}")
    
    return merge_history

In [36]:
history = iterative_cka_merge_and_train(
    model=model,
    train_dataloader=train_dataloader,
    val_dataloader=val_dataloader,
    task_name='mrpc',
    device=device,
    cka_evaluator=cka_eval,
    tracker=tracker,
    init_metric=init_metric,
    num_merges=6,
    target_layers=[6, 7, 8],  # Save models with 6, 7, 8 layers
    recovery_epochs=3,
    recovery_lr=1e-5,
    save_dir='./weights_test/'
)

Target Metric for mrpc: f1
Original f1 score: 0.9048
Recovery threshold: 0.0090

Merge Iteration: 1/6

[1/5] Computing CKA Similarity...


CKA Evaluation: 100%|████████████████████████████████████████████████████████████████| 115/115 [00:43<00:00,  2.62it/s]


  Similarity Stats:
    Average: 0.906263
    Highest: 0.996595
    Lowest:  0.751576

[2/5] Merging Layers...
  Merged Layers: 9 + 10
  Layer Composition: [[0], [1], [2], [3], [4], [5], [6], [7], [8], [9, 10], [11]]
  Total Layers: 11

[3/5] Evaluating Post-Merge Performance...
  Metrics:
      accuracy: 0.8627
    ★ f1: 0.9064

[4/5] Recovery Training...
  Performance drop: -0.0015 (threshold: 0.0090)
  → Recovery training NOT needed

[5/5] Saving Model...
  • 11 layers (not in target list, skipping save)

Merge Iteration: 2/6

[1/5] Computing CKA Similarity...


CKA Evaluation: 100%|████████████████████████████████████████████████████████████████| 115/115 [00:52<00:00,  2.18it/s]


  Similarity Stats:
    Average: 0.895910
    Highest: 0.979809
    Lowest:  0.752902

[2/5] Merging Layers...
  Merged Layers: 6 + 7
  Layer Composition: [[0], [1], [2], [3], [4], [5], [6, 7], [8], [9, 10], [11]]
  Total Layers: 10

[3/5] Evaluating Post-Merge Performance...
  Metrics:
      accuracy: 0.8186
    ★ f1: 0.8818

[4/5] Recovery Training...
  Performance drop: 0.0231 (threshold: 0.0090)
  → Recovery training NEEDED
  → Training: 3 epochs, lr=1e-05


...

  Similarity Stats:
    Average: 0.830846
    Highest: 0.942694
    Lowest:  0.659738

[2/5] Merging Layers...
  Merged Layers: 6 + 7
  Layer Composition: [[0], [1, 2], [3], [4], [5], [6, 7], [8, 9, 10, 11]]
  Total Layers: 7

[3/5] Evaluating Post-Merge Performance...
  Metrics:
      accuracy: 0.8701
    ★ f1: 0.9069

[4/5] Recovery Training...
  Performance drop: -0.0020 (threshold: 0.0090)
  → Recovery training NOT needed

[5/5] Saving Model...
  ✓ Saved 7-layer model to ./weights_test/layer-7-mrpc.pt

Merge Iteration: 6/6

[1/5] Computing CKA Similarity...




CKA Evaluation:   0%|                                                                          | 0/115 [00:00<?, ?it/s]

CKA Evaluation:   1%|▌                                                                 | 1/115 [00:00<01:18,  1.45it/s]

CKA Evaluation:   2%|█▏                                                                | 2/115 [00:01<01:05,  1.73it/s]

CKA Evaluation:   3%|█▋                                                                | 3/115 [00:01<01:01,  1.82it/s]

CKA Evaluation:   3%|██▎                                                               | 4/115 [00:02<01:14,  1.49it/s]

CKA Evaluation:   4%|██▊                                                               | 5/115 [00:03<01:09,  1.58it/s]

CKA Evaluation:   5%|███▍                                                              | 6/115 [00:03<01:04,  1.68it/s]

CKA Evaluation:   6%|████                                                              | 7/115 [00:04<01:14,  1.46it/s]

CKA Evaluation:   7%|████▌    

  Similarity Stats:
    Average: 0.808646
    Highest: 0.938712
    Lowest:  0.658971

[2/5] Merging Layers...
  Merged Layers: 4 + 5
  Layer Composition: [[0], [1, 2], [3], [4], [5, 6, 7], [8, 9, 10, 11]]
  Total Layers: 6

[3/5] Evaluating Post-Merge Performance...
  Metrics:
      accuracy: 0.8137
    ★ f1: 0.8694

[4/5] Recovery Training...
  Performance drop: 0.0354 (threshold: 0.0090)
  → Recovery training NEEDED
  → Training: 3 epochs, lr=1e-05


Training:   0%|          | 0/345 [00:00<?, ?it/s]

✓ Saved checkpoint (best loss: 0.4688)

<----------------- Epoch 1 ----------------->
Loss: 0.21, Training Metrics:
  accuracy: 0.9198
  f1: 0.9401
Validation Loss: 0.47, Validation Metrics:
  accuracy: 0.8260
  f1: 0.8790
Elapsed Time: 384.6161 sec

<----------------- Epoch 2 ----------------->
Loss: 0.13, Training Metrics:
  accuracy: 0.9545
  f1: 0.9661
Validation Loss: 0.49, Validation Metrics:
  accuracy: 0.8382
  f1: 0.8842
Elapsed Time: 396.5288 sec

<----------------- Epoch 3 ----------------->
Loss: 0.1, Training Metrics:
  accuracy: 0.9700
  f1: 0.9776
Validation Loss: 0.54, Validation Metrics:
  accuracy: 0.8309
  f1: 0.8808
Elapsed Time: 388.6088 sec

Total Training Time: 1172.29 sec
Best validation loss: 0.4688

  → Loading best checkpoint...
  ✓ Loaded best checkpoint from epoch 1
    Best val loss: 0.4688
    Best val metrics: {'accuracy': 0.8259803921568627, 'f1': 0.879045996592845}
  ✓ Cleaned up temporary checkpoint

[5/5] Saving Model...
  ✓ Saved 6-layer model to ./

In [37]:
# See what happened
print("Merge history:")
for i, iteration in enumerate(history['iterations']):
    print(f"  Iteration {iteration['iteration']}:")
    print(f"    Merged: {iteration['merged_layers']}")
    print(f"    Layers after: {iteration['num_layers_after']}")
    print(f"    Composition: {history['layer_compositions'][i]}")
    
    if history['training_stats'][i]:
        final_metrics = history['metrics_after_recovery'][i]
        print(f"    Final metrics: {final_metrics}")

Merge history:
  Iteration 1:
    Merged: (9, 10)
    Layers after: 11
    Composition: [[0], [1, 2], [3], [4], [5, 6, 7], [8, 9, 10, 11]]
  Iteration 2:
    Merged: (6, 7)
    Layers after: 10
    Composition: [[0], [1, 2], [3], [4], [5, 6, 7], [8, 9, 10, 11]]
    Final metrics: {'accuracy': 0.8602941176470589, 'f1': 0.901213171577123}
  Iteration 3:
    Merged: (8, 9)
    Layers after: 9
    Composition: [[0], [1, 2], [3], [4], [5, 6, 7], [8, 9, 10, 11]]
  Iteration 4:
    Merged: (1, 2)
    Layers after: 8
    Composition: [[0], [1, 2], [3], [4], [5, 6, 7], [8, 9, 10, 11]]
    Final metrics: {'accuracy': 0.8578431372549019, 'f1': 0.8986013986013986}
  Iteration 5:
    Merged: (6, 7)
    Layers after: 7
    Composition: [[0], [1, 2], [3], [4], [5, 6, 7], [8, 9, 10, 11]]
  Iteration 6:
    Merged: (4, 5)
    Layers after: 6
    Composition: [[0], [1, 2], [3], [4], [5, 6, 7], [8, 9, 10, 11]]
    Final metrics: {'accuracy': 0.8259803921568627, 'f1': 0.879045996592845}
